Membuat user retention cohort di Python

Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

In [ ]:
import os
os.getcwd()

'/content'

Import data dari CSV ke dataframe

In [ ]:
df = pd.read_csv('Online Retail Data.csv', header=0)
df


,order_id,product_code,product_name,quantity,order_date,price,customer_id
0,493410,TEST001,This is a test product.,5,1/4/2010 9:24,4.50,12346.0
1,C493411,21539,RETRO SPOTS BUTTER DISH,-1,1/4/2010 9:43,4.25,14590.0
2,493412,TEST001,This is a test product.,5,1/4/2010 9:53,4.50,12346.0
3,493413,21724,PANDA AND BUNNIES STICKER SHEET,1,1/4/2010 9:54,0.85,NaN
4,493413,84578,ELEPHANT TOY WITH BLUE T-SHIRT,1,1/4/2010 9:54,3.75,NaN
...,...,...,...,...,...,...,...
461768,539991,21618,4 WILDFLOWER BOTANICAL CANDLES,1,12/23/2010 16:49,1.25,NaN
461769,539991,72741,GRAND CHOCOLATECANDLE,4,12/23/2010 16:49,1.45,NaN
461770,539992,21470,FLOWER VINE RAFFIA FOOD COVER,1,12/23/2010 17:41,3.75,NaN
461771,539992,22258,FELT FARM ANIMAL RABBIT,1,12/23/2010 17:41,1.25,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 461773 entries, 0 to 461772
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      461773 non-null  object 
 1   product_code  461773 non-null  object 
 2   product_name  459055 non-null  object 
 3   quantity      461773 non-null  int64  
 4   order_date    461773 non-null  object 
 5   price         461773 non-null  float64
 6   customer_id   360853 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 24.7+ MB


Data Cleansing

In [ ]:
df_clean = df.copy()
# membuat kolom date
df_clean['date'] = pd.to_datetime(df_clean['order_date']).dt.date.astype('datetime64[ns]')
# menghapus semua baris tanpa customer_id
df_clean = df_clean[~df_clean['customer_id'].isna()]
# menghapus semua baris tanpa product_name
df_clean = df_clean[~df_clean['product_name'].isna()]
# membuat semua product_name berhuruf kecil
df_clean['product_name'] = df_clean['product_name'].str.lower()
# menghapus semua baris dengan product_code atau product_name test
# membuat kolom order_status dengan nilai 'cancelled' jika order_id diawali dengan huruf 'c' dan 'delivered' jika order_id tanpa awalan huruf 'c'
df_clean['order_status'] = np.where(df_clean['order_id'].str[:1]=='C', 'cancelled', 'delivered')
# mengubah nilai quantity yang negatif menjadi positif karena nilai negatif tersebut hanya menandakan order tersebut cancelled
df_clean['quantity'] = df_clean['quantity'].abs()
# menghapus baris dengan price bernilai negatif
df_clean = df_clean[df_clean['price']>0]
# membuat nilai amount, yaitu perkalian antara quantity dan price
df_clean['amount'] = df_clean['quantity'] * df_clean['price']
# mengganti product_name dari product_code yang memiliki beberapa product_name dengan salah satu product_name-nya yang paling sering muncul
most_freq_product_name = df_clean.groupby(['product_code','product_name'], as_index=False).agg(order_cnt=('order_id','nunique')).sort_values(['product_code','order_cnt'], ascending=[True,False])
most_freq_product_name['rank'] = most_freq_product_name.groupby('product_code')['order_cnt'].rank(method='first', ascending=False)
most_freq_product_name = most_freq_product_name[most_freq_product_name['rank']==1].drop(columns=['order_cnt','rank'])
df_clean = df_clean.merge(most_freq_product_name.rename(columns={'product_name':'most_freq_product_name'}), how='left', on='product_code')
df_clean['product_name'] = df_clean['most_freq_product_name']
df_clean = df_clean.drop(columns='most_freq_product_name')
# mengkonversi customer_id menjadi string
df_clean['customer_id'] = df_clean['customer_id'].astype(str)
# menghapus outlier
from scipy import stats
df_clean = df_clean[(np.abs(stats.zscore(df_clean[['quantity','amount']]))<3).all(axis=1)]
df_clean = df_clean.reset_index(drop=True)
df_clean

,order_id,product_code,product_name,quantity,order_date,price,customer_id,date,order_status,amount
0,493410,TEST001,this is a test product.,5,1/4/2010 9:24,4.50,12346.0,2010-01-04,delivered,22.50
1,C493411,21539,red retrospot butter dish,1,1/4/2010 9:43,4.25,14590.0,2010-01-04,cancelled,4.25
2,493412,TEST001,this is a test product.,5,1/4/2010 9:53,4.50,12346.0,2010-01-04,delivered,22.50
3,493414,21844,red retrospot mug,36,1/4/2010 10:28,2.55,14590.0,2010-01-04,delivered,91.80
4,493414,21533,retro spot large milk jug,12,1/4/2010 10:28,4.25,14590.0,2010-01-04,delivered,51.00
...,...,...,...,...,...,...,...,...,...,...
358473,539988,84380,set of 3 butterfly cookie cutters,1,12/23/2010 16:06,1.25,18116.0,2010-12-23,delivered,1.25
358474,539988,84849D,hot baths soap holder,1,12/23/2010 16:06,1.69,18116.0,2010-12-23,delivered,1.69
358475,539988,84849B,fairy soap soap holder,1,12/23/2010 16:06,1.69,18116.0,2010-12-23,delivered,1.69
358476,539988,22854,cream sweetheart egg holder,2,12/23/2010 16:06,4.95,18116.0,2010-12-23,delivered,9.90


In [ ]:
df_clean.head(10)

,order_id,product_code,product_name,quantity,order_date,price,customer_id,date,order_status,amount
0,493410,TEST001,this is a test product.,5,1/4/2010 9:24,4.50,12346.0,2010-01-04,delivered,22.50
1,C493411,21539,red retrospot butter dish,1,1/4/2010 9:43,4.25,14590.0,2010-01-04,cancelled,4.25
2,493412,TEST001,this is a test product.,5,1/4/2010 9:53,4.50,12346.0,2010-01-04,delivered,22.50
3,493414,21844,red retrospot mug,36,1/4/2010 10:28,2.55,14590.0,2010-01-04,delivered,91.80
4,493414,21533,retro spot large milk jug,12,1/4/2010 10:28,4.25,14590.0,2010-01-04,delivered,51.00
5,493414,37508,new england ceramic cake server,2,1/4/2010 10:28,2.55,14590.0,2010-01-04,delivered,5.10
6,493414,35001G,hand open shape gold,2,1/4/2010 10:28,4.25,14590.0,2010-01-04,delivered,8.50
7,493414,21527,red retrospot traditional teapot,12,1/4/2010 10:28,6.95,14590.0,2010-01-04,delivered,83.40
8,493414,21531,red retrospot sugar jam bowl,24,1/4/2010 10:28,2.10,14590.0,2010-01-04,delivered,50.40
9,C493415,21527,red retrospot traditional teapot,3,1/4/2010 10:33,7.95,14590.0,2010-01-04,cancelled,23.85


In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 358478 entries, 0 to 358477
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      358478 non-null  object        
 1   product_code  358478 non-null  object        
 2   product_name  358478 non-null  object        
 3   quantity      358478 non-null  int64         
 4   order_date    358478 non-null  object        
 5   price         358478 non-null  float64       
 6   customer_id   358478 non-null  object        
 7   date          358478 non-null  datetime64[ns]
 8   order_status  358478 non-null  object        
 9   amount        358478 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(6)
memory usage: 27.3+ MB


MEMBUAT RFM SEGMENTATION

Agregat data transaksi ke bentuk summary total transaksi (order), total nilai order(order value), tanggal order terakhir dari setiap pengguna

In [ ]:
df_user = df_clean.groupby('customer_id', as_index= False).agg(order_cnt=('order_id', 'nunique'),
                                                              max_order_date=('date', 'max'),
                                                              total_order_value=('amount', 'sum'))
df_user

,customer_id,order_cnt,max_order_date,total_order_value
0,12346.0,10,2010-10-04,696.90
1,12608.0,1,2010-10-31,415.79
2,12745.0,2,2010-08-10,723.85
3,12746.0,2,2010-06-30,266.35
4,12747.0,19,2010-12-13,4094.79
...,...,...,...,...
3885,18283.0,6,2010-11-22,641.77
3886,18284.0,2,2010-10-06,486.68
3887,18285.0,1,2010-02-17,427.00
3888,18286.0,2,2010-08-20,941.48


Kolom jumlah hari sejak order terakhir

In [ ]:
today = df_clean['date'].max()
df_user['day_since_last_order'] = (today - df_user['max_order_date']).dt.days
df_user

,customer_id,order_cnt,max_order_date,total_order_value,day_since_last_order
0,12346.0,10,2010-10-04,696.90,80
1,12608.0,1,2010-10-31,415.79,53
2,12745.0,2,2010-08-10,723.85,135
3,12746.0,2,2010-06-30,266.35,176
4,12747.0,19,2010-12-13,4094.79,10
...,...,...,...,...,...
3885,18283.0,6,2010-11-22,641.77,31
3886,18284.0,2,2010-10-06,486.68,78
3887,18285.0,1,2010-02-17,427.00,309
3888,18286.0,2,2010-08-20,941.48,125


In [ ]:
df_user.describe()

,order_cnt,max_order_date,total_order_value,day_since_last_order
count,3890.000000,3890,3890.000000,3890.000000
mean,5.129563,2010-09-23 05:57:35.629820160,1544.260713,90.751671
min,1.000000,2010-01-05 00:00:00,1.250000,0.000000
25%,1.000000,2010-08-19 00:00:00,296.165000,25.000000
50%,3.000000,2010-10-26 00:00:00,648.300000,58.000000
75%,6.000000,2010-11-28 00:00:00,1585.617500,126.000000
max,163.000000,2010-12-23 00:00:00,71970.390000,352.000000
std,8.498706,NaN,3434.453016,88.820754


Buat binning dari jumlah hari sejak order terakhir yang terdiri dari 5 bins dengan batas-batasnya merupakan min, P20, P40, P60, P80, max dan beri label 1 sampai 5 dari bin tertinggi ke terendah sebagai skor recency

In [ ]:
df_user['recency_score'] = pd.cut(df_user['day_since_last_order'],
                                  bins=[df_user['day_since_last_order'].min(),
                                        np.percentile(df_user['day_since_last_order'], 20),
                                        np.percentile(df_user['day_since_last_order'], 40),
                                        np.percentile(df_user['day_since_last_order'], 60),
                                        np.percentile(df_user['day_since_last_order'], 80),
                                        df_user['day_since_last_order'].max()],
                                  labels=[5, 4, 3, 2, 1],
                                  include_lowest=True).astype(int)
df_user


,customer_id,order_cnt,max_order_date,total_order_value,day_since_last_order,recency_score
0,12346.0,10,2010-10-04,696.90,80,2
1,12608.0,1,2010-10-31,415.79,53,3
2,12745.0,2,2010-08-10,723.85,135,2
3,12746.0,2,2010-06-30,266.35,176,1
4,12747.0,19,2010-12-13,4094.79,10,5
...,...,...,...,...,...,...
3885,18283.0,6,2010-11-22,641.77,31,4
3886,18284.0,2,2010-10-06,486.68,78,2
3887,18285.0,1,2010-02-17,427.00,309,1
3888,18286.0,2,2010-08-20,941.48,125,2


In [ ]:
df_user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3890 entries, 0 to 3889
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   customer_id           3890 non-null   object        
 1   order_cnt             3890 non-null   int64         
 2   max_order_date        3890 non-null   datetime64[ns]
 3   total_order_value     3890 non-null   float64       
 4   day_since_last_order  3890 non-null   int64         
 5   recency_score         3890 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(1)
memory usage: 182.5+ KB


Buat binning dari total transaksi (order) yang terdiri dari 5 bins dengan batas-batasnya merupakan min, P20, P40, P60, P80, max dan beri label 1 sampai 5 dari bin terendah ke tertinggi sebagai skor frequency

In [ ]:
df_user['frequency_score'] = pd.cut(df_user['order_cnt'],
                                    bins=[0,
                                          np.percentile(df_user['order_cnt'], 20),
                                          np.percentile(df_user['order_cnt'], 40),
                                          np.percentile(df_user['order_cnt'], 60),
                                          np.percentile(df_user['order_cnt'], 80),
                                          df_user['order_cnt'].max()],
                                    labels=[1, 2, 3, 4, 5],
                                    include_lowest=True).astype(int)
df_user

,customer_id,order_cnt,max_order_date,total_order_value,day_since_last_order,recency_score,frequency_score
0,12346.0,10,2010-10-04,696.90,80,2,5
1,12608.0,1,2010-10-31,415.79,53,3,1
2,12745.0,2,2010-08-10,723.85,135,2,2
3,12746.0,2,2010-06-30,266.35,176,1,2
4,12747.0,19,2010-12-13,4094.79,10,5,5
...,...,...,...,...,...,...,...
3885,18283.0,6,2010-11-22,641.77,31,4,4
3886,18284.0,2,2010-10-06,486.68,78,2,2
3887,18285.0,1,2010-02-17,427.00,309,1,1
3888,18286.0,2,2010-08-20,941.48,125,2,2


In [ ]:
df_user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3890 entries, 0 to 3889
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   customer_id           3890 non-null   object        
 1   order_cnt             3890 non-null   int64         
 2   max_order_date        3890 non-null   datetime64[ns]
 3   total_order_value     3890 non-null   float64       
 4   day_since_last_order  3890 non-null   int64         
 5   recency_score         3890 non-null   int64         
 6   frequency_score       3890 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(4), object(1)
memory usage: 212.9+ KB


Buat binning dari total nilai order (order value) yang terdiri dari 5 bins dengan batas-batasnya merupakan min, P20, P40, P60, P80, max dan beri label 1 sampai 5 dari bin terendah ke tertinggi sebagai skor monetary

In [ ]:
df_user['monetary_score'] = pd.cut(df_user['total_order_value'],
                                   bins=[df_user['total_order_value'].min(),
                                         np.percentile(df_user['total_order_value'], 20),
                                         np.percentile(df_user['total_order_value'], 40),
                                         np.percentile(df_user['total_order_value'], 60),
                                         np.percentile(df_user['total_order_value'], 80),
                                         df_user['total_order_value'].max()],
                                   labels=[1, 2, 3, 4, 5],
                                   include_lowest=True).astype(int)
df_user

,customer_id,order_cnt,max_order_date,total_order_value,day_since_last_order,recency_score,frequency_score,monetary_score
0,12346.0,10,2010-10-04,696.90,80,2,5,3
1,12608.0,1,2010-10-31,415.79,53,3,1,2
2,12745.0,2,2010-08-10,723.85,135,2,2,3
3,12746.0,2,2010-06-30,266.35,176,1,2,2
4,12747.0,19,2010-12-13,4094.79,10,5,5,5
...,...,...,...,...,...,...,...,...
3885,18283.0,6,2010-11-22,641.77,31,4,4,3
3886,18284.0,2,2010-10-06,486.68,78,2,2,3
3887,18285.0,1,2010-02-17,427.00,309,1,1,2
3888,18286.0,2,2010-08-20,941.48,125,2,2,4


In [ ]:
df_user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3890 entries, 0 to 3889
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   customer_id           3890 non-null   object        
 1   order_cnt             3890 non-null   int64         
 2   max_order_date        3890 non-null   datetime64[ns]
 3   total_order_value     3890 non-null   float64       
 4   day_since_last_order  3890 non-null   int64         
 5   recency_score         3890 non-null   int64         
 6   frequency_score       3890 non-null   int64         
 7   monetary_score        3890 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(5), object(1)
memory usage: 243.3+ KB


kolom nama segmen berdasarkan skor recency dan frequency

In [ ]:
df_user['segment'] = np.select(
    [(df_user['recency_score']==5) & (df_user['frequency_score']>=4),
     (df_user['recency_score'].between(3, 4)) & (df_user['frequency_score']>=4),
     (df_user['recency_score']>=4) & (df_user['frequency_score'].between(2, 3)),
     (df_user['recency_score']<=2) & (df_user['frequency_score']==5),
     (df_user['recency_score']==3) & (df_user['frequency_score']==3),
     (df_user['recency_score']==5) & (df_user['frequency_score']==1),
     (df_user['recency_score']==4) & (df_user['frequency_score']==1),
     (df_user['recency_score']<=2) & (df_user['frequency_score'].between(3, 4)),
     (df_user['recency_score']==3) & (df_user['frequency_score']<=2),
     (df_user['recency_score']<=2) & (df_user['frequency_score']<=2)],
    ['01-Champion', '02-Loyal Customers', '03-Potential Loyalists', "04-Can't Lose Them", '05-Need Attention',
     '06-New Customers', '07-Promising', '08-At Risk', '09-About to Sleep', '10-Hibernating'],
    default='Unknown Segment'
)
df_user

,customer_id,order_cnt,max_order_date,total_order_value,day_since_last_order,recency_score,frequency_score,monetary_score,segment
0,12346.0,10,2010-10-04,696.90,80,2,5,3,04-Can't Lose Them
1,12608.0,1,2010-10-31,415.79,53,3,1,2,09-About to Sleep
2,12745.0,2,2010-08-10,723.85,135,2,2,3,10-Hibernating
3,12746.0,2,2010-06-30,266.35,176,1,2,2,10-Hibernating
4,12747.0,19,2010-12-13,4094.79,10,5,5,5,01-Champion
...,...,...,...,...,...,...,...,...,...
3885,18283.0,6,2010-11-22,641.77,31,4,4,3,02-Loyal Customers
3886,18284.0,2,2010-10-06,486.68,78,2,2,3,10-Hibernating
3887,18285.0,1,2010-02-17,427.00,309,1,1,2,10-Hibernating
3888,18286.0,2,2010-08-20,941.48,125,2,2,4,10-Hibernating


In [ ]:
df_user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3890 entries, 0 to 3889
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   customer_id           3890 non-null   object        
 1   order_cnt             3890 non-null   int64         
 2   max_order_date        3890 non-null   datetime64[ns]
 3   total_order_value     3890 non-null   float64       
 4   day_since_last_order  3890 non-null   int64         
 5   recency_score         3890 non-null   int64         
 6   frequency_score       3890 non-null   int64         
 7   monetary_score        3890 non-null   int64         
 8   segment               3890 non-null   object        
dtypes: datetime64[ns](1), float64(1), int64(5), object(2)
memory usage: 273.6+ KB


Tampilkan summary dari RFM segmentation (poin 8) berupa banyaknya pengguna, rata-rata dan median dari total order, total order value, dan jumlah hari sejak order terakhir

In [ ]:
summary = pd.pivot_table(df_user, index='segment',
               values=['customer_id','day_since_last_order','order_cnt','total_order_value'],
               aggfunc={'customer_id': pd.Series.nunique,
                        'day_since_last_order': [np.mean, np.median],
                        'order_cnt': [np.mean, np.median],
                        'total_order_value': [np.mean, np.median]})
summary['pct_unique'] = (summary['customer_id'] / summary['customer_id'].sum() * 100).round(1)
summary

/tmp/ipython-input-690746973.py:1: FutureWarning: The provided callable <function mean at 0x7c9e9c94b880> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  summary = pd.pivot_table(df_user, index='segment',
/tmp/ipython-input-690746973.py:1: FutureWarning: The provided callable <function median at 0x7c9e99d9bba0> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  summary = pd.pivot_table(df_user, index='segment',


customer_id day_since_last_order         order_cnt  \
                           nunique                 mean median       mean   
segment                                                                     
01-Champion                    550            10.618182    9.5  15.467273   
02-Loyal Customers             546            40.864469   37.0   8.767399   
03-Potential Loyalists         523            23.573614   24.0   2.829828   
04-Can't Lose Them              65           121.338462  112.0  11.353846   
05-Need Attention              176            58.613636   59.0   3.397727   
06-New Customers                50            14.220000   16.0   1.000000   
07-Promising                   142            32.760563   34.0   1.000000   
08-At Risk                     426           140.455399  120.0   4.136150   
09-About to Sleep              352            58.735795   58.0   1.417614   
10-Hibernating                1060           196.667925  198.5   1.312264   

                              total_order_value           pct_unique  
                       median              mean    median             
segment                                                               
01-Champion              10.0       5003.674245  2775.525       14.1  
02-Loyal Customers        7.0       2622.817826  1946.850       14.0  
03-Potential Loyalists    3.0        766.769828   622.070       13.4  
04-Can't Lose Them       10.0       2806.978154  2267.350        1.7  
05-Need Attention         3.0        989.232676   826.370        4.5  
06-New Customers          1.0        244.689000   193.675        1.3  
07-Promising              1.0        287.800282   238.440        3.7  
08-At Risk                4.0       1152.500918   875.430       11.0  
09-About to Sleep         1.0        448.229688   334.755        9.0  
10-Hibernating            1.0        343.086154   257.005       27.2

* pengguna paling banyak berada pada segmen Hibernating (1060 atau 27.3%), champion (550 atau 14.1%), dan loyal customers (546 atau 14.0%).
* program khusus fokus pada urgensi bertransaksi untuk loyal customers (546 atau 14.0%) dapat dibuat untuk membuat mereka bertransaksi kembali dalam waktu dekat sehingga bisa naik ke segmen champion.
* program khusus yang fokus pada jumlah transaksi untuk potential loyalitas (523 atau 13.4%) dapat dibuat untuk membuat mereka lebih sering bertransaksi sehingga bisa naik ke segmen champion.
* program khusus untuk hibernating (1060 atau 27.3%) dapat dibuat untuk membuat mereka kembali bertransaksi walaupun belum begitu sering sehingga bisa naik ke segmen new customers atau bahkan potential loyalists.